### Import packages, define paths and load data

In [132]:
from pathlib import Path
import json
import spacy
import re
import gender_guesser.detector as gender
d = gender.Detector()

nlp = spacy.load("en_core_web_sm")

In [133]:
# use the system's built-in word list (available on Mac/Linux without downloading)
with open('/usr/share/dict/words', 'r') as f:
    english_words = set(w.strip().lower() for w in f)

In [145]:
# Define project and data paths
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/cleaned"
OUTPUT_DIR = PROJECT_ROOT / "data/gendered_names"

sci_fi_data_path = DATA_DIR / "sci_fi_stories_cleaned.jsonl"
romance_data_path = DATA_DIR / "romance_stories_cleaned.jsonl"
literary_fiction_data_path = DATA_DIR / "lit_fiction_stories_cleaned.jsonl"

In [135]:
def load_jsonl(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

In [136]:
lit_fic_stories = load_jsonl(literary_fiction_data_path)
romance_stories = load_jsonl(romance_data_path)
sci_fi_stories = load_jsonl(sci_fi_data_path)

In [137]:
# make small subset of sci_fi_stories
subset = sci_fi_stories[:100]

In [138]:
def is_common_word(name):
    name_lower = name.lower()
    # check singular and plural forms
    return (name_lower in english_words or 
            name_lower.rstrip('s') in english_words)  # remove plural s

def validate_gender(name, pronoun_gender):
    if is_common_word(name) and d.get_gender(name) == "unknown":
        return False
    
    result = d.get_gender(name)
    if result in ("unknown", "andy"):
        return True
    if pronoun_gender == "male" and result in ("male", "mostly_male"):
        return True
    if pronoun_gender == "female" and result in ("female", "mostly_female"):
        return True
    return False

In [139]:
male_pronouns = {"he", "him", "his"}
female_pronouns = {"she", "her", "hers"}

def clean_name(name):
    # extract only the first clean capitalized word
    match = re.search(r'[A-Z][a-z]+', name)
    return match.group(0) if match else None

def extract_gendered_names(stories):
    male_associated = set()
    female_associated = set()
    conflicted = set()

    for story in stories:
        text = story.get('text', str(story)).replace('\\n', '\n')
        doc = nlp(text)

        for sent in doc.sents:
            sent_lower = set(w.lower() for w in sent.text.split())
            has_male = bool(sent_lower & male_pronouns)
            has_female = bool(sent_lower & female_pronouns)

            if not (has_male ^ has_female):
                continue

            names = set()
            for ent in sent.ents:
                if ent.label_ == "PERSON":
                    clean = clean_name(ent.text.split()[0])
                    if clean:
                        names.add(clean)

            for name in names:
                if has_male and validate_gender(name, "male"):
                    if name in female_associated:
                        conflicted.add(name)
                        female_associated.discard(name)
                    elif name not in conflicted:
                        male_associated.add(name)
                elif has_female and validate_gender(name, "female"):
                    if name in male_associated:
                        conflicted.add(name)
                        male_associated.discard(name)
                    elif name not in conflicted:
                        female_associated.add(name)

    return sorted(list(male_associated)), sorted(list(female_associated)), sorted(list(conflicted))


In [140]:
male_associated_sci_fi, female_associated_sci_fi, conflicted_sci_fi = extract_gendered_names(sci_fi_stories)

In [141]:
male_associated_lit, female_associated_lit, conflicted_lit = extract_gendered_names(lit_fic_stories)

In [142]:
male_associated_romance, female_associated_romance, conflicted_romance = extract_gendered_names(romance_stories)

In [177]:
# manually clean the lists
# removing surnames, cities and non-names
words_to_remove = ["Activating", "Afterburn", "Aftervault", "Ascension", "Attuner", "Albuquerque", "Aristotle", "Aftershave", "Ashford", "Ackerman", "Alvarado", "Archivum", "Arkpark",
                   "Borrowed", "Backyard", "Barbernet", "Biolab", "Bay", "Biblioteca", "Brightwater", "Buenos", "Bentley", "Bellemore", "Bellini", "Betadine", "Box", "Barnes", "Boardroom", "Brighton", "Buongiorno", "Burnham", "Bosphorus", "Bramwell", "Brylcreem", "Baudelaire", "Bellford", "Bloomfield", "Bront", "Brubeck",
                   "Chainlock", "Chordlocks", "Chronotech", "Complicit", "Cellmate", "Cappadocia", "Calderon", "Caplan", "Cavanaugh", "Cipressi", "Costello", "Calderman", "Callaghan", "Chang", "Carmichael", "Chaucer", "Carvalho", "Castillo", "Cruz", "Corbett",
                   "Deploying", "Driftline", "Dalloway", "Delaney", "Dickinson", "Dalrymple", "Delacroix", "Devereaux", "Diaz", "Delgado", "Devereux", "Dewitt", "Dobrev", "Duvall", "Derryhood", "Dostoevsky", "Dostoyevsky", "Dvor", "Debussy",
                   "Eberhart", "Earthrise", "Echocert", "Elbridge", "Elmwood", "Ellman", "Echelion", "Ennik",
                   "Floor", "Foundries", "Frostline", "Foryou", "Freud", "Fitzgerald", "Faulkner", "Feet", "Fareeha", "Fordham", "Friedman", "Fernandes",
                   "Gatewatch", "Goodnight", "Grayhaven", "Glasswings", "Gardner", "Grantham", "Graybridge", "Greenway", "Gelsomini", "Graywatch", "Grierson", "Gravelle", "Gellman", "Gerber", "Glasgow", "Goldman", "Gammapier", "Glassfawn", "Gavrien", "Grosvenor", "Gustalink",
                   "Harmonized", "Hearthlight", "Hearthline", "Hollowpoint", "Harborline", "Harwick", "Hanford", "Harborwick", "Harrington", "Harper", "Hawke", "Hawthorne", "Headteacher", "Henderson", "Holdfasttime", "Hollis", "Halverson", "Hartwell", "Hellinger", "Harding", "Holbrook", "Halberg", "Halverton", "Hartfold", "Hensley", "Holloway", "Higgins", "Hargreeve", "Harrigan", "Harwood", "Hatfield", "Helvetica", "Hexham", "Holcomb", "Halstead", "Halvek", "Haversham", "Hoffmann", "Hollowell",
                   "Instagram",
                   "Jackpoint", "Jazzmaster", "Jackpot", "Jansen", "Jefferson", "Jung", "Jovanovic",
                   "Keepline", "Kintsugi", "Koenig", "Kincaid", "Kjellfjord", "Kerrigan", "Kessler", "Kavinsky", "Kershaw", "Kominsky", "Klein", "Kaczynski", "Koivunen", "Kowalski",
                   "Loopback", "Lightcall", "Langley", "Langdon", "Leclerc", "Leland", "Lopez", "Lagrange", "Langston", "Linton", "Llewellyn", "Lockhart", "Lenhart", "Lerner", "Levitt", "Lenton", "Lindstrom", "Larkfield",
                   "Metadata", "Mmm", "Mnemoscientist", "Moonpool", "Mc", "Midservice", "Montefiore", "Monteverde", "Maplewood", "Marshall", "Mendez", "Minced", "Monroe", "Marchand", "Mandelstam", "Markham", "Morrisons", "Myrtle", "Menelaus",
                   "Newsfeeds", "Nudging", "Nocturnarium", "Negroni", "Nanogel", "Naples", "Nabokov", "Nevins", "Netwatch",
                   "Omniframe", "Onboard", "Onfoot",
                   "Peacekeepers", "Plexiglas", "Patience", "Paralegal", "Palmer", "Pascal", "Perez", "Pernambuco", "Price", "Plato", "Pritchard", "Pritchett", "Poulenc", "Piper", "Papi", "Paladino",
                   "Rainwater", "Remembered", "Rootroom", "Riverline", "Rachmaninoff", "Riverton", "Rodriguez", "Randall", "Rorschach", "Rotterdam", "Rianova", "Rinaldi", "Robinson", "Rosenthal",
                   "Skydrift", "Sleepline", "Stackhouse", "Sundering", "Skywall", "Skillcaster", "Salinger", "Seabright", "Seafar", "Seafoam", "Skyraven", "Soundcheck", "Stearman", "Stonehaven", "Storycut", "Steinway", "Storycuts", "Surprised", "Shakespeare", "Strutline", "Syntelife", "Salvatore", "Schubert", "Sinatra", "Strauss", "Steele", "Seiler", "Sibelius", "Steinbeck", "Smirnov", "Stradivex", "Sverdsen",
                   "Tennessee", "Torres", "Teenager", "Tomlin", "Tenebris", "Tectonix",
                   "Vossbacher", "Visconti",
                   "Winterspine", "Whitcomb", "Whitby", "Wakecraft", "Whitaker", "Wintersong", "Winterfield", "Winnicott", "Wirt", "Willoughby",
                   "You",
                   "Zookeeper", "Ziploc"]



In [178]:
words_to_remove_set = set(words_to_remove)

lists = [
    "male_associated_romance",
    "female_associated_romance",
    "male_associated_lit",
    "female_associated_lit",
    "male_associated_sci_fi",
    "female_associated_sci_fi"
]

for name in lists:
    globals()[name] = [w for w in globals()[name] if w not in words_to_remove_set]

In [179]:
# Save lists
with open(OUTPUT_DIR / "male_names_sci_fi.json", "w") as f:
    json.dump(male_associated_sci_fi, f)

with open(OUTPUT_DIR / "female_names_sci_fi.json", "w") as f:
    json.dump(female_associated_sci_fi, f)

with open(OUTPUT_DIR / "male_names_lit.json", "w") as f:
    json.dump(male_associated_lit, f)   

with open(OUTPUT_DIR / "female_names_lit.json", "w") as f:
    json.dump(female_associated_lit, f)

with open(OUTPUT_DIR / "male_names_romance.json", "w") as f:
    json.dump(male_associated_romance, f)

with open(OUTPUT_DIR / "female_names_romance.json", "w") as f:
    json.dump(female_associated_romance, f)